# SOEN 471 - SmartCart Assignment Notebook

## Part 1 - Data PreProcessing

In [ ]:
# Load data
import pandas as pd

user_data = pd.read_csv('data/ecommerce_user_data.csv')
product_data = pd.read_csv('data/product_details.csv')

print(user_data.head())
print(product_data.head())

In [ ]:
# Clean the data (its already pretty clean)

# --- user_data ---

# 1. Fix data types
user_data['Timestamp'] = pd.to_datetime(user_data['Timestamp'], errors='coerce')

# 2. Standardize strings
user_data['UserID']    = user_data['UserID'].str.strip()
user_data['ProductID'] = user_data['ProductID'].str.strip()
user_data['Category']  = user_data['Category'].str.strip().str.lower()

# 3. Standardize string columns
user_data['Category'] = user_data['Category'].str.strip().str.lower()
user_data['UserID']  = user_data['UserID'].astype(str).str.strip()

# 4. Validate rating range (should be 1–5)
invalid_ratings = user_data[~user_data['Rating'].between(1, 5)]
print(f"Invalid ratings: {len(invalid_ratings)}")
user_data = user_data[user_data['Rating'].between(1, 5)]

# 5. Drop duplicates (same user rating same product twice)
before = len(user_data)
user_data = user_data.drop_duplicates(subset=['UserID', 'ProductID'])
print(f"Duplicate user-product pairs removed: {before - len(user_data)}")

# --- product_data ---

# 1. Standardize strings
product_data['ProductID']   = product_data['ProductID'].str.strip()
product_data['ProductName'] = product_data['ProductName'].str.strip()
product_data['Category']    = product_data['Category'].str.strip().str.lower()

# 2. Drop duplicate products
before = len(product_data)
product_data = product_data.drop_duplicates(subset=['ProductID'])
print(f"Duplicate products removed: {before - len(product_data)}")

# --- Summary ---
print(f"\nuser_data shape:    {user_data.shape}")
print(f"product_data shape: {product_data.shape}")
print("\nuser_data dtypes:\n",    user_data.dtypes)
print("\nproduct_data dtypes:\n", product_data.dtypes)

In [ ]:
# Convert into user-item matrix
user_item_matrix = user_data.pivot_table(
        values='Rating',
        index='UserID',
        columns='ProductID',
        fill_value=0
    )

print(user_item_matrix.head())

In [ ]:
# Group by user and category
agg_user_data = user_data.groupby(['UserID', 'Category'])['Rating'].agg(
    AverageRating='mean',
    Count='count'
)

print(agg_user_data.head(20))

## Part 2 - User-Based Collaborative Filtering (Cosine Similarity) 

In [ ]:
# Compute cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
from seaborn import heatmap

similarity_matrix = cosine_similarity(user_item_matrix)
similarity_df = pd.DataFrame(similarity_matrix, index=user_item_matrix.index, columns=user_item_matrix.index)
print(similarity_df.head())

heatmap(similarity_df)


In [ ]:
def recommend_products(target_user, user_item_matrix, similarity_df, top_n_users=5, top_n_products=5):
    # 1. Get top n similar users (excluding the user themselves)
    similar_users = similarity_df[target_user].sort_values(ascending=False)[1:top_n_users+1].index
    
    # 2. Get products the target user hasn't rated yet (value is 0)
    target_user_ratings = user_item_matrix.loc[target_user]
    unrated_products = target_user_ratings[target_user_ratings == 0].index
    
    # 3. Calculate weighted average scores for unrated products
    product_scores = {}
    for product in unrated_products:
        weights = 0
        score = 0
        for sim_user in similar_users:
            # Check if the similar user has rated this product
            sim_rating = user_item_matrix.loc[sim_user, product]
            if sim_rating > 0:
                similarity = similarity_df.loc[target_user, sim_user]
                score += similarity * sim_rating
                weights += similarity
        
        if weights > 0:
            product_scores[product] = score / weights
        else:
            product_scores[product] = 0
            
    # 4. Sort and return the top N recommendations
    recommendations = sorted(product_scores.items(), key=lambda x: x[1], reverse=True)[:top_n_products]
    return recommendations

# Example usage for User U000
recs = recommend_products('U000', user_item_matrix, similarity_df)

if recs:
    recs_df = (
        pd.DataFrame(recs, columns=['ProductID', 'Score'])
          .merge(product_data[['ProductID', 'ProductName', 'Category']], on='ProductID', how='left')
          .sort_values('Score', ascending=False)
          .reset_index(drop=True)
    )
    print("Top recommendations for U000:")
    print(recs_df.to_string(index=False))
else:
    print("No recommendations for U000.")

In [ ]:
# Implement evaluation metrics
from sklearn.metrics import precision_score

# Simplified Precision@K check
# See if the recommended products' categories match the user's top categories
def evaluate_recs(target_user, recommendations, user_data, product_data):
    user_fav_categories = user_data[user_data['UserID'] == target_user]['Category'].unique()
    
    hits = 0
    for prod_id, score in recommendations:
        category = product_data[product_data['ProductID'] == prod_id]['Category'].values[0]
        if category in user_fav_categories:
            hits += 1
            
    precision_at_k = hits / len(recommendations) if recommendations else 0
    return precision_at_k

print(f"Precision@5 for U000: {evaluate_recs('U000', recs, user_data, product_data)}")


## Part 3 - Association Rule Mining (Apriori)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations, chain
import networkx as nx
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from typing import List, Dict, Tuple, Set

In [ ]:
# Convert each user-product interations into a transaction
# Transaction = {UserID: set/list of ProductIDs}
transactions = user_data.groupby('UserID')['ProductID'].apply(list).tolist()
transactions = [t for t in transactions if t] # Exclude empty transactions

product_name_map = dict(zip(product_data['ProductID'], product_data['ProductName']))
def get_product_names(product_ids):
    return [product_name_map.get(pid, pid) for pid in product_ids]

print(f"\nTransaction Statistics:")
print(f"  Total transactions: {len(transactions)}")
print(f"  Average transaction size: {np.mean([len(t) for t in transactions]):.2f} items")
print(f"  Max transaction size: {max([len(t) for t in transactions])} items")
print(f"  Min transaction size: {min([len(t) for t in transactions])} items")

# Sample transactions
print("\nSample Transactions:")
for i, trans in enumerate(transactions[:5], 1):
    print(f"  Transaction {i}: {trans[:5]}{'...' if len(trans) > 5 else ''}")

In [ ]:
# Encode transactions using TransactionEncoder
# Apriori algo in mlxtend needs one-hot encoding
# Each row -> 1 transaction & each column -> 1 product
# Value = 0 is product not in transaction & value = 1 is product in transaction

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_ary, columns=te.columns_)

print(f"\nTransaction df shape: {df_trans.shape}")
print(f"Number of unique products: {df_trans.shape[1]}")
print(f"\nTransaction df preview:")
print(df_trans.iloc[:5, :10])

In [ ]:
# This notebook cell takes a few minutes to run (approximately 5 min)
# Minimum support threshold -- Can change
MIN_SUPPORT = 0.01

# Applying Apriori algo to find frequent itemset
frequent_itemsets = apriori(df_trans, min_support=MIN_SUPPORT, use_colnames=True)

# Display frequent itemsets
print(f"\nFound {len(frequent_itemsets)} frequent itemsets")
print(f"\nFrequent itemsets preview:")
print(frequent_itemsets.head(10))

In [ ]:
# Convert frozenset to readable format (list)
frequent_itemsets['itemsets_list'] = frequent_itemsets['itemsets'].apply(list)
frequent_itemsets['itemsets_str'] = frequent_itemsets['itemsets'].apply(lambda x: ', '.join(x))

print("\nFrequent itemsets - Readable format:")
print(frequent_itemsets[['support', 'itemsets_str']].head())

In [ ]:
# Frequent itemsets' size distribution
itemset_sizes = frequent_itemsets['itemsets'].apply(len)
print(f"\nItemset size distribution:")
size_dist = itemset_sizes.value_counts().sort_index()
for size, count in size_dist.items():
    print(f"  Size {size}: {count} itemsets")

In [ ]:
# Largest itemsets
frequent_itemsets['size'] = frequent_itemsets['itemsets'].apply(len)

print(f"\nLargest itemsets (by number of items):")
largest_itemsets = frequent_itemsets.nlargest(5, 'size')
for i, row in largest_itemsets.iterrows():
    items = ', '.join(list(row['itemsets'])[:5])
    if len(row['itemsets']) > 5:
        items += f" and {len(row['itemsets'])-5} more..."
    print(f"  Support: {row['support']:.4f} | Items: {items}")

In [ ]:
# Association rules
MIN_CONFIDENCE = 0.5

# W/out this, a large num of rules is generated - hard to analyze and takes a long time to load
frequent_itemsets_small = frequent_itemsets[frequent_itemsets['size'] <= 3].copy()
rules = association_rules(frequent_itemsets_small, metric='confidence', min_threshold=MIN_CONFIDENCE)

# Display rules
print("\nUsing only itemsets of size <= 3 for rule generation:")
print(f"Generated {len(rules)} association rules")
print(f"Min Confidence: {MIN_CONFIDENCE}")
print("\nTop 10 Association Rules by lift:")
rules_sorted = rules.sort_values('lift', ascending=False)
display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage']
print(rules_sorted[display_cols].head(10).to_string())

print(f"\nRule Statistics:")
print(f"  Average lift: {rules['lift'].mean():.4f}")
print(f"  Max lift: {rules['lift'].max():.4f}")
print(f"  Average confidence: {rules['confidence'].mean():.4f}")
print(f"  Average support: {rules['support'].mean():.4f}")

In [ ]:
# Visualizations of frequent itemsets
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Get top 15 itemsets by support (only size 1-3)
top_itemsets = frequent_itemsets_small.nlargest(15, 'support').copy()

# Labels
top_itemsets['label'] = top_itemsets['itemsets_str'].apply(
    lambda x: x[:40] + '...' if len(x) > 40 else x
)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(top_itemsets)), top_itemsets['support'].values, 
               color=plt.cm.viridis(np.linspace(0, 1, len(top_itemsets))))

ax.set_yticks(range(len(top_itemsets)))
ax.set_yticklabels(top_itemsets['label'].values, fontsize=10)
ax.set_xlabel('Support', fontsize=12, fontweight='bold')
ax.set_ylabel('Itemset', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Frequent Itemsets by Support', fontsize=14, fontweight='bold')
ax.invert_yaxis() 

# Labels
for i, (bar, support) in enumerate(zip(bars, top_itemsets['support'].values)):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{support:.3f}', va='center', fontsize=9)

# Grid
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"\nTop itemset: {top_itemsets.iloc[0]['itemsets_str']}")
print(f"   Support: {top_itemsets.iloc[0]['support']:.3f} ({top_itemsets.iloc[0]['support']*100:.1f}%)")

In [ ]:
# Bar chart of itemset size distribution (only sizes 1-3) 
size_counts = frequent_itemsets_small['size'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars = ax.bar(size_counts.index, size_counts.values, color=colors, edgecolor='black', alpha=0.8)

ax.set_xlabel('Itemset Size', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Frequent Itemsets', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Frequent Itemset Sizes (Size ≤ 3)', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Size 1\n(Single items)', 'Size 2\n(Item pairs)', 'Size 3\n(Triple items)'])

# Labels
for bar, count in zip(bars, size_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count:,}', ha='center', fontsize=11, fontweight='bold')

total_itemsets = len(frequent_itemsets_small)
ax.text(0.5, 0.95, f'Total: {total_itemsets:,} frequent itemsets (size ≤ 3)',
        transform=ax.transAxes, ha='center', fontsize=11, 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"  Size 1 itemsets: {size_counts[1]:,} (most frequent single items)")
print(f"  Size 2 itemsets: {size_counts[2]:,} (product pairs)")
print(f"  Size 3 itemsets: {size_counts[3]:,} (product triples)")
